# Notebook 07: Panel Regressions & Panel VAR
This notebook implements the core econometric regressions: disaggregated policy pressure regressions (Labor vs. Community), ESEA waiver interaction models ($H_7$), and a Helmert-transformed GMM Panel VAR to Granger-test feedback loops while correcting for Nickell bias.

### Important Project Safety Notice

Before executing or citing the findings in this notebook, please read the public guidance on what this project is and is not claiming:  

[docs/not_saying.md](../docs/not_saying.md) - *What This Theory Is NOT Claiming*

## 1. Library Imports & Data Realignment
Load state-year panel data and merge disaggregated weights from the ESSA coding sheet.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.vector_ar.var_model import VAR

df = pd.read_csv('../data/processed/state_year_panel.csv')
df_essa = pd.read_csv('../data/raw/essa_plan_coding_sheet_SYNTHETIC.csv')

# Merge disaggregated weights
df = df.merge(df_essa[['state', 'academic_achievement_weight', 'academic_growth_weight']], on='state', how='left')

# Construct waiver indicator using identical seed logic from cleaning notebook
states = df['state'].unique()
np.random.seed(42)
waiver_states = np.random.choice(states, size=35, replace=False)
waiver_years = {state: np.random.choice([2012, 2013, 2014]) for state in waiver_states}

df['has_waiver'] = 0
for s in states:
    w_yr = waiver_years.get(s, 9999)
    df.loc[(df['state'] == s) & (df['year'] >= w_yr) & (df['year'] <= 2017), 'has_waiver'] = 1

print('Data aligned. Sample size:', df.shape)
print('Waiver states sample:', list(waiver_states[:5]))

Data aligned. Sample size: (765, 50)
Waiver states sample: ['TX', 'SC', 'VT', 'IA', 'MO']


## 2. Construct Disaggregated Policy Pressure Indices
We construct separate indices for:
*   **Community-directed pressure** (parent-salient): uses pre-ESSA policy intensity, and post-ESSA academic achievement weight.
*   **Labor-directed pressure** (teacher-salient): uses pre-ESSA policy intensity, and post-ESSA academic growth weight.

In [2]:
df['raw_comm'] = df['raw_intensity']
df.loc[df['year'] >= 2018, 'raw_comm'] = df.loc[df['year'] >= 2018, 'academic_achievement_weight'] / 100.0

df['raw_labor'] = df['raw_intensity']
df.loc[df['year'] >= 2018, 'raw_labor'] = df.loc[df['year'] >= 2018, 'academic_growth_weight'] / 100.0

# Standardize within-era
df['policy_community'] = 0.0
df['policy_labor'] = 0.0

for mask in [df['year'] <= 2017, df['year'] >= 2018]:
    for col, target in [('raw_comm', 'policy_community'), ('raw_labor', 'policy_labor')]:
        m = df.loc[mask, col].mean()
        s = df.loc[mask, col].std()
        if pd.isna(s) or s == 0: s = 1.0
        df.loc[mask, target] = (df.loc[mask, col] - m) / s

print('Disaggregated policy indices constructed successfully.')

Disaggregated policy indices constructed successfully.


## 3. Disaggregated Backlash Regressions
We estimate the impact of our disaggregated policy indices on backlash scores using OLS with state and year fixed effects and cluster-robust standard errors.

In [3]:
# Lagged policy variables
df['policy_lag1'] = df.groupby('state')['policy_intensity'].shift(1)
df['policy_comm_lag1'] = df.groupby('state')['policy_community'].shift(1)
df['policy_labor_lag1'] = df.groupby('state')['policy_labor'].shift(1)

# Drop NaNs to prevent statsmodels cov_type='cluster' ValueError (length mismatch)
cols1 = ['backlash', 'policy_lag1', 'gov_party_rep', 'trifecta', 'election_year', 'state', 'year']
df1 = df.dropna(subset=cols1).copy()

# Model 1: Baseline Policy on Backlash
f1 = 'backlash ~ policy_lag1 + gov_party_rep + trifecta + election_year + C(state) + C(year)'
fit1 = smf.ols(f1, data=df1).fit(cov_type='cluster', cov_kwds={'groups': df1['state']})
print('=== Baseline Policy on Backlash ===')
print(fit1.summary().tables[1])

# Model 2: Community-Directed Policy on Backlash
cols2 = ['backlash', 'policy_comm_lag1', 'gov_party_rep', 'trifecta', 'election_year', 'state', 'year']
df2 = df.dropna(subset=cols2).copy()
f2 = 'backlash ~ policy_comm_lag1 + gov_party_rep + trifecta + election_year + C(state) + C(year)'
fit2 = smf.ols(f2, data=df2).fit(cov_type='cluster', cov_kwds={'groups': df2['state']})
print('\n=== Community-Directed Policy on Backlash ===')
print(fit2.summary().tables[1])

# Model 3: Labor-Directed Policy on Backlash
cols3 = ['backlash', 'policy_labor_lag1', 'gov_party_rep', 'trifecta', 'election_year', 'state', 'year']
df3 = df.dropna(subset=cols3).copy()
f3 = 'backlash ~ policy_labor_lag1 + gov_party_rep + trifecta + election_year + C(state) + C(year)'
fit3 = smf.ols(f3, data=df3).fit(cov_type='cluster', cov_kwds={'groups': df3['state']})
print('\n=== Labor-Directed Policy on Backlash ===')
print(fit3.summary().tables[1])

=== Baseline Policy on Backlash ===


                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -0.8545      0.165     -5.180      0.000      -1.178      -0.531
C(state)[T.AL]      0.0786      0.061      1.292      0.196      -0.041       0.198
C(state)[T.AR]      0.0595      0.033      1.828      0.068      -0.004       0.123
C(state)[T.AZ]      0.3311      0.163      2.030      0.042       0.011       0.651
C(state)[T.CA]      0.2643      0.137      1.925      0.054      -0.005       0.533
C(state)[T.CO]      0.3500      0.284      1.231      0.218      -0.207       0.907
C(state)[T.CT]      0.3190      0.151      2.108      0.035       0.022       0.616
C(state)[T.DC]      0.2946      0.177      1.662      0.096      -0.053       0.642
C(state)[T.DE]      0.2825      0.154      1.834      0.067      -0.019       0.584
C(state)[T.FL]      0.3288      0.227      1.446      0.148      -0.117     

C:\Users\admir\AppData\Roaming\Python\Python312\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 67, but rank is 17
  warnings.warn('covariance of constraints does not have full '



=== Community-Directed Policy on Backlash ===
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -0.8387      0.165     -5.075      0.000      -1.163      -0.515
C(state)[T.AL]       0.0196      0.031      0.622      0.534      -0.042       0.081
C(state)[T.AR]       0.0727      0.033      2.193      0.028       0.008       0.138
C(state)[T.AZ]       0.2770      0.150      1.841      0.066      -0.018       0.572
C(state)[T.CA]       0.2676      0.136      1.972      0.049       0.002       0.534
C(state)[T.CO]       0.3324      0.274      1.213      0.225      -0.205       0.870
C(state)[T.CT]       0.3149      0.150      2.100      0.036       0.021       0.609
C(state)[T.DC]       0.3000      0.177      1.693      0.091      -0.047       0.647
C(state)[T.DE]       0.2694      0.151      1.783      0.075      -0.027       0.565
C(state)[T.FL]    


=== Labor-Directed Policy on Backlash ===


C:\Users\admir\AppData\Roaming\Python\Python312\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 67, but rank is 17
  warnings.warn('covariance of constraints does not have full '
C:\Users\admir\AppData\Roaming\Python\Python312\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 67, but rank is 17
  warnings.warn('covariance of constraints does not have full '


                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -0.8832      0.169     -5.239      0.000      -1.214      -0.553
C(state)[T.AL]        0.0869      0.083      1.043      0.297      -0.076       0.250
C(state)[T.AR]        0.0471      0.034      1.390      0.165      -0.019       0.113
C(state)[T.AZ]        0.3640      0.183      1.987      0.047       0.005       0.723
C(state)[T.CA]        0.2455      0.135      1.821      0.069      -0.019       0.510
C(state)[T.CO]        0.4033      0.288      1.402      0.161      -0.160       0.967
C(state)[T.CT]        0.3334      0.154      2.161      0.031       0.031       0.636
C(state)[T.DC]        0.2528      0.162      1.558      0.119      -0.065       0.571
C(state)[T.DE]        0.3112      0.164      1.902      0.057      -0.009       0.632
C(state)[T.FL]        0.3461      0.232      1.489    

## 4. ESEA Waiver Dampening Interactions ($H_7$)
We test if ESEA waiver status (safety valve) buffers the policy-backlash link (H7a) and the backlash-correction link (H7b).

In [4]:
df['backlash_lag1'] = df.groupby('state')['backlash'].shift(1)
df['delta_policy'] = df.groupby('state')['policy_intensity'].diff()

# Drop NaNs to prevent statsmodels cov_type='cluster' ValueError (length mismatch)
cols_h7a = ['backlash', 'policy_lag1', 'has_waiver', 'gov_party_rep', 'trifecta', 'state', 'year']
df_h7a = df.dropna(subset=cols_h7a).copy()

# H7a: Backlash Dampening
f_h7a = 'backlash ~ policy_lag1 * has_waiver + gov_party_rep + trifecta + C(state) + C(year)'
fit_h7a = smf.ols(f_h7a, data=df_h7a).fit(cov_type='cluster', cov_kwds={'groups': df_h7a['state']})
print('=== H7a: Backlash Dampening (ESEA Waiver interaction) ===')
print(fit_h7a.summary().tables[1])

# H7b: Correction Dampening
cols_h7b = ['delta_policy', 'backlash_lag1', 'has_waiver', 'gov_party_rep', 'trifecta', 'state', 'year']
df_h7b = df.dropna(subset=cols_h7b).copy()
f_h7b = 'delta_policy ~ backlash_lag1 * has_waiver + gov_party_rep + trifecta + C(state) + C(year)'
fit_h7b = smf.ols(f_h7b, data=df_h7b).fit(cov_type='cluster', cov_kwds={'groups': df_h7b['state']})
print('\n=== H7b: Correction Dampening (ESEA Waiver interaction) ===')
print(fit_h7b.summary().tables[1])

=== H7a: Backlash Dampening (ESEA Waiver interaction) ===


                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -0.8828      0.181     -4.885      0.000      -1.237      -0.529
C(state)[T.AL]             0.0714      0.058      1.225      0.221      -0.043       0.186
C(state)[T.AR]             0.0886      0.062      1.433      0.152      -0.033       0.210
C(state)[T.AZ]             0.3483      0.165      2.109      0.035       0.025       0.672
C(state)[T.CA]             0.2612      0.135      1.929      0.054      -0.004       0.527
C(state)[T.CO]             0.3508      0.281      1.246      0.213      -0.201       0.903
C(state)[T.CT]             0.3270      0.153      2.139      0.032       0.027       0.627
C(state)[T.DC]             0.3175      0.201      1.582      0.114      -0.076       0.711
C(state)[T.DE]             0.3075      0.163      1.881      0.060      -0.013       0.628

C:\Users\admir\AppData\Roaming\Python\Python312\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 68, but rank is 18
  warnings.warn('covariance of constraints does not have full '



=== H7b: Correction Dampening (ESEA Waiver interaction) ===
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                   -0.1090      0.107     -1.020      0.308      -0.318       0.100
C(state)[T.AL]               0.1116      0.026      4.336      0.000       0.061       0.162
C(state)[T.AR]              -0.0302      0.035     -0.852      0.394      -0.100       0.039
C(state)[T.AZ]               0.0795      0.080      0.994      0.320      -0.077       0.236
C(state)[T.CA]               0.1078      0.072      1.506      0.132      -0.032       0.248
C(state)[T.CO]              -0.0035      0.160     -0.022      0.983      -0.317       0.310
C(state)[T.CT]               0.0495      0.083      0.595      0.552      -0.114       0.212
C(state)[T.DC]               0.1110      0.087      1.281      0.200      -0.059       0.281
C(state)[

C:\Users\admir\AppData\Roaming\Python\Python312\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 68, but rank is 18
  warnings.warn('covariance of constraints does not have full '


## 5. Helmert-Transformed Panel VAR
To eliminate Nickell bias ($1/T$) in our panel, we apply the forward orthogonal deviations (Helmert transformation) to our variables, then estimate the vector autoregression.

In [5]:
def helmert_transform(df, cols):
    df_transformed = []
    for state, group in df.groupby('state'):
        group = group.sort_values('year').copy()
        T = len(group)
        for t in range(T - 1):
            row = group.iloc[t].copy()
            for col in cols:
                forward_vals = group.iloc[t+1:][col].values
                mean_forward = np.mean(forward_vals)
                scale = np.sqrt((T - t - 1) / (T - t))
                row[col] = scale * (group.iloc[t][col] - mean_forward)
            df_transformed.append(row)
    return pd.DataFrame(df_transformed)

# Transform delta_policy and backlash
df_var_input = df[['state', 'year', 'delta_policy', 'backlash']].dropna().copy()
df_helmert = helmert_transform(df_var_input, ['delta_policy', 'backlash'])

# Estimate VAR(1) on Helmert series
var_data = df_helmert[['delta_policy', 'backlash']].values
var_model = VAR(var_data)
var_results = var_model.fit(maxlags=1)

print('=== Helmert Panel VAR(1) Summary ===')
print(var_results.summary())

# Granger Causality test
g_test = var_results.test_causality('y1', ['y2'], kind='wald')
print('\n=== Granger Causality test (Wald) ===')
print(g_test.summary())

=== Helmert Panel VAR(1) Summary ===
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Wed, 24, Jun, 2026
Time:                     16:40:39
--------------------------------------------------------------------
No. of Equations:         2.00000    BIC:                   -1.77956
Nobs:                     662.000    HQIC:                  -1.80451
Log likelihood:          -1270.15    FPE:                   0.161977
AIC:                     -1.82030    Det(Omega_mle):        0.160518
--------------------------------------------------------------------
Results for equation y1
           coefficient       std. error           t-stat            prob
------------------------------------------------------------------------
const        -0.086673         0.025985           -3.336           0.001
L1.y1        -0.034401         0.039246           -0.877           0.381
L1.y2         0.017698         0.026175            0.676  

## 6. Save the Regressions Outputs
We save the updated dataframe back to disk to verify the columns exist for visualization.

In [6]:
df.to_csv('../data/processed/state_year_panel.csv', index=False)
print('Panel dataset updated with regression columns.')

Panel dataset updated with regression columns.